# LINK drill — finish the chain (A-level shape)

This is not a resit of any prelim. The behaviours are the ones that drop marks on an A-level task ladder: a subtask is unfinished until its test has run, the empty/miss/first case is part of the answer, the next part must use the previous return, and every name is copied from this stem.

**Shape (TYS 2020–2025, not the 2026 warehouse/workshop mix):** file in → ordered records → search whose index is consumed by the next function → linked structure with an empty-list case → one web page that shows a result.

**Rules**
- Do not open `P2 CHAIN A/B/C/D` or their answers.
- Checkpoints are at the bottom. Answers are only in `_archive/LINK drill - answers.ipynb`.
- A part is done when you have executed it and the printed result is in this notebook.
- Before coding each part, copy the identifiers below into your cell. Do not rename them.

Data: `LINK_BAYS.csv`. L4 has no HTML files. You write `templates/link/desk.html` and `templates/link/lane.html` from the L4 stem.


---
## L1 — load and reject (file task)

`LoadBays("LINK_BAYS.csv")` returns two lists:

- `clean`: `(code, vessel, berth, tonnes:int)` 
- `rejected`: one short reason string per bad row

A row is clean only when `code` and `vessel` are non-empty and `tonnes` is an integer.

**Test you must run and leave visible:** print `len(clean)`, `len(rejected)`, and `clean[0]`.

Keep the name `clean`.


In [7]:
# L1 — YOUR ATTEMPT
# identifiers: LoadBays, clean, rejected

clean = []
rejected = []

import csv

def LoadBays(filename):
    with open(filename) as f:
        file = csv.reader(f)
        next(file)
        for row in file:
            if not row[0]:
                rejected.append('missing code')
            elif not row[1]:
                rejected.append('missing vessel')
            elif not row[3].isnumeric():
                rejected.append('tonnes not int')
            else:
                row[3] = int(row[3])
                clean.append(tuple(row))

LoadBays('LINK_BAYS.csv')
print(len(clean), len(rejected), clean[0])

7 2 ('A12', 'North Star', '3', 400)


---
## L2 — order, then search, then use the index

Sort `clean` into `ordered` by `code` ascending, then `vessel` ascending. Any correct sort is fine. Do not call Python's `sorted` or `list.sort`.

`FindFirst(ordered, code)` returns the index of the **first** row with that code, or `-1`. The window must shrink. A miss must return `-1` rather than loop.

`ListVessels(ordered, code)` must call `FindFirst`. From that index, collect vessels while the code matches. If `FindFirst` returns `-1`, return `[]`. Do not scan from index 0.

**Test you must run:** 
- `FindFirst(ordered, "B07")` and `FindFirst(ordered, "Z99")`
- `print(ListVessels(ordered, "B07"))`
- `print(ListVessels(ordered, "Z99"))`


In [11]:
# L2 — YOUR ATTEMPT
# identifiers: ordered, FindFirst, ListVessels

ordered = []

#we shall use a mergesort because im feeling like it

def mergevessels(arr, first,mid, last):
    left = arr[first : mid + 1]
    right = arr[mid + 1 : last]
    l = 0
    r = 0
    a = first
    #compare heads and fill main arr with smaller head
    while l < len(left) and r < len(right):
        if left[l][0] == right[r][0]:
            #compare vessel when code is the same
            if left[l][1] > right[r][1]:
                #right branch head vessel smaller; put right
                arr[a] = right[r]
                a += 1
                r += 1
            else:
                arr[a] = left[l]
                a += 1
                l += 1
        elif left[l][0] > right[r][0]:
            arr[a] = right[r]
            a += 1
            r += 1
        else:
            arr[a] = left[l]
            a += 1
            l += 1
    #fill leftovers
    while l < len(left):
        arr[a] = left[l]
        a += 1
        l += 1
    while r < len(right):
        arr[a] = right[r]
        a += 1
        r += 1
    return arr

def mergesortvessels(array, first, last):
    if first < last:
        mid = (first + last) // 2
        mergesortvessels(array, first, mid )
        mergesortvessels(array, mid + 1, last)
        mergevessels(array, first, mid, last)
    return array

cl = clean
ordered = mergesortvessels(cl, 0, len(cl)-1)

def FindFirst(ordered, code):
    index = 0
    for item in ordered:
        if item[0] == code:
            return index
        index += 1
    return -1 

def ListVessels(ordered, code):
    found = FindFirst(ordered,code)
    collect = []
    for item in ordered[found:]:
        if item[0] == code:
            collect.append(item[1])
        else:
            return collect

print(
    ordered, '\n',
    FindFirst(ordered,'B07'), '\n',
    FindFirst(ordered, 'Z99'), '\n',
    ListVessels(ordered, 'B07'), '\n',
    ListVessels(ordered, 'Z99')
)

[('A12', 'North Star', '3', 400), ('A12', 'South Wind', '3', 410), ('B07', 'Amber', '1', 200), ('B07', 'Coral', '1', 180), ('B07', 'Delta', '2', 190), ('C01', 'Echo', '5', 900), ('D04', 'Quay', '4', 50)] 
 2 
 -1 
 ['Amber', 'Coral', 'Delta'] 
 []


---
## L3 — insert into an empty chain, then in front

`Call(ref, vessel)` stores private `ref` and `vessel`, with `getRef`, `getVessel`, `getNext`, `setNext`.

`Lane` starts empty (`getFirst` is `None`). `AddCall(new_call)` inserts by ascending `ref`:

- empty lane: the new call becomes first
- smaller than first: it becomes the new first, and the old first follows it
- duplicate `ref`: print `Duplicate ref` and insert nothing
- otherwise: insert between two calls, or at the end

`Brief()` walks with `getNext` and returns a list of `(ref, vessel)`.

Keep the object in `bay_lane` so it does not clash with the later route function.

**Test you must run, in this order:** add ref `30` to an empty lane, add `10` (must become first), add `20`, add `10` again, then `print(bay_lane.Brief())`.


In [3]:
# L3 — YOUR ATTEMPT
# identifiers: Call, Lane, bay_lane

bay_lane = None

class Call:
    def __init__(self, ref, vessel):
        self.__ref = ref
        self.__vessel = vessel
        self.__next = None
        
    def getRef(self):
        return self.__ref
    
    def getVessel(self):
        return self.__vessel
    
    def getNext(self):
        return self.__next
    
    def setNext(self, n):
        self.__next = n
        
class Lane:
    def __init__(self):
        self.__first = None
        
    def getFirst(self):
        return self.__first
    
    def AddCall(self, new_call):
        if self.__first is None:
            self.__first = new_call
            return
        elif new_call.getRef() < self.__first.getRef():
            new_call.setNext(self.__first)
            self.__first = new_call
            return
        previous = self.__first
        current = previous.getNext()
        while current:
            if previous.getRef() == new_call.getRef():
                print('duplicate ref')
                return
            elif current.getRef() > new_call.getRef():
                #insert when new is smaller than the current one
                previous.setNext(new_call)
                new_call.setNext(current)
                return 
            previous = current
            current = current.getNext()
        #current is none and we have to insert at the end 
        previous.setNext(new_call)
        return
    
    def Brief(self):
        fin = []
        current = self.__first
        while current:
            fin.append((current.getRef(), current.getVessel()))
            current = current.getNext()
        return fin
    
bay_lane = Lane()
bay_lane.AddCall(Call(30, 'orion'))
bay_lane.AddCall(Call(10, 'saggitarus'))
bay_lane.AddCall(Call(20, 'centaurus'))
bay_lane.AddCall(Call(10, 'sirius'))
print(bay_lane.Brief())
        

duplicate ref
[(10, 'saggitarus'), (20, 'centaurus'), (30, 'orion')]


---
## L4 — routes, and both pages written from this stem

Create the two files yourself. The names below are the whole specification.

`templates/link/desk.html`
- a heading
- the message in `error` when that variable is set
- a form that POSTs to the `desk` route
- one text field named `berth_code`, and a submit control

`templates/link/lane.html`
- a heading that shows `berth_code`
- one line per item in `rows`: the vessel and the berth
- a visible empty result when `rows` has no items
- a link back to the `desk` route

Routes:

- `desk` on `/`, GET and POST. POST reads `request.form` key `berth_code`. If it is non-empty, **return** `redirect(url_for("lane", berth_code=...))`. Otherwise show `desk.html` with `error`.
- `lane` on `/lane/<berth_code>`. From `ordered`, keep rows whose berth equals that code. `render_template("link/lane.html", rows=..., berth_code=...)`.
- Do not reuse route or field names from an older Flask task.
- `app.run` is not required for the checkpoint.

**Test you must run:** print the rows your `lane` filter keeps for berth `"1"`. The checkpoint then POSTs `berth_code` `1` and reads the lane page.


In [ ]:
# L4 — YOUR ATTEMPT
# identifiers: app, desk, lane_rows
from flask import Flask, render_template, request, redirect, url_for

app = Flask(__name__)
lane_rows = []

ac = ['GET', 'POST']

@app.route('/', methods = ac)
def desk():
    error = None
    if request.method == 'POST':
        berth_code = request.form.get('berth_code','')
        if berth_code:
            return redirect(url_for('lane', berth_code = berth_code))
        error = 'Please fill in all fields'
    return render_template('link/desk.html', error = error)

@app.route('/lane/<berth_code>', methods = ac)
def lane(berth_code):
    rows = []
    for i in ordered:
        if i[2] == berth_code:
            rows.append(i)
    return render_template('link/lane.html', berth_code = berth_code, rows = rows)

if __name__ == '__main__':
    print(ordered)
    app.run(port=5004)

[('A12', 'North Star', '3', 400), ('A12', 'South Wind', '3', 410), ('B07', 'Amber', '1', 200), ('B07', 'Coral', '1', 180), ('B07', 'Delta', '2', 190), ('C01', 'Echo', '5', 900), ('D04', 'Quay', '4', 50)]
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5004
Press CTRL+C to quit
127.0.0.1 - - [28/Sep/2026 20:39:43] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 20:39:43] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [28/Sep/2026 20:39:46] "POST / HTTP/1.1" 302 -
127.0.0.1 - - [28/Sep/2026 20:39:46] "GET /lane/3 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 20:39:46] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [28/Sep/2026 20:39:47] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 20:39:47] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [28/Sep/2026 20:39:49] "POST / HTTP/1.1" 302 -
127.0.0.1 - - [28/Sep/2026 20:39:49] "GET /lane/1 HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 20:39:49] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [28/Sep/2026 20:39:50] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [28/Sep/2026 20:39:50] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [28/Sep/2026 20:39:52] "POST / HTTP/1.1" 302 -
127.0.0.1 - - [28/Sep/2026 20:39:52] "GET /lane/2 HTTP/1.1" 200 -
12

---
# Checkpoints

Run after the attempts. A FAIL here means that part has no evidence yet.


In [12]:
# CHECKPOINTS — LINK
PARTS = ("L1", "L2", "L3", "L4")


def _check_l1():
    if len(clean) != 7:
        return False, f"clean has {len(clean)}, expected 7"
    if len(rejected) < 2:
        return False, "rejected rows missing"
    if clean[0][0] == "" or not isinstance(clean[0][3], int):
        return False, "first clean row is not (code, vessel, berth, int tonnes)"
    return True, "ok"


def _check_l2():
    if FindFirst(ordered, "Z99") != -1:
        return False, "miss must return -1"
    idx = FindFirst(ordered, "B07")
    if idx < 0 or ordered[idx][0] != "B07":
        return False, "FindFirst did not land on B07"
    if idx > 0 and ordered[idx - 1][0] == "B07":
        return False, "FindFirst must return the first B07, not a later one"
    got = ListVessels(ordered, "B07")
    if got != ["Amber", "Coral", "Delta"]:
        return False, f"ListVessels(B07) is {got}"
    if ListVessels(ordered, "Z99") != []:
        return False, "missing code must return an empty list"
    return True, "ok"


def _check_l3():
    brief = bay_lane.Brief()
    refs = [r for r, _ in brief]
    if refs != [10, 20, 30]:
        return False, f"Brief refs {refs}; empty-lane and insert-before-first are required"
    if any(v == "" for _, v in brief):
        return False, "vessel missing"
    return True, "ok"


def _check_l4():
    if not lane_rows or any(str(r[2]) != "1" for r in lane_rows):
        return False, "lane_rows must be the berth 1 rows from ordered"
    client = app.test_client()
    posted = client.post("/", data={"berth_code": "1"})
    if posted.status_code not in (301, 302):
        return False, f"POST desk returned {posted.status_code}; berth_code must redirect"
    loc = posted.headers.get("Location", "")
    if "/lane/1" not in loc:
        return False, f"redirect went to {loc}"
    page = client.get("/lane/1")
    if page.status_code != 200 or b"Amber" not in page.data:
        return False, "lane did not render the berth 1 rows"
    return True, "ok"


_CHECKS = {"L1": _check_l1, "L2": _check_l2, "L3": _check_l3, "L4": _check_l4}
for part in PARTS:
    try:
        ok, detail = _CHECKS[part]()
    except Exception as e:
        ok, detail = False, str(e)
    print(f"{part}: {'PASS' if ok else 'FAIL — ' + detail}")


L1: PASS
L2: PASS
L3: PASS
L4: FAIL — lane_rows must be the berth 1 rows from ordered
